In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import ToTensor
from PIL import Image
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import random
import os

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

# ===== 하이퍼파라미터 =====
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
batch_size = 8
img_size = 512
heatmap_size = 128
lr = 2e-4
num_epochs = 1000
num_genes = 15

marker_genes = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]

# ===== 경로 설정 =====
data_dir = '../../data/marker_gene_spatial_expression'
model_path = '../../model/marker_gene_spatial_expression/'
create_dir(model_path)

print(f'Device: {device}')
print(f'Batch size: {batch_size}')
print(f'Learning rate: {lr}')
print(f'Input: 20x {img_size}x{img_size} + 5x {img_size}x{img_size} (multi-scale)')
print(f'Output: {num_genes}x{heatmap_size}x{heatmap_size}')

In [ ]:
# ===== Multi-Scale Dataset =====
class MultiScaleDataset(Dataset):
    def __init__(self, img_20x_list, img_5x_list, label_list, augment=False):
        self.img_20x_paths = img_20x_list
        self.img_5x_paths = img_5x_list
        self.label_paths = label_list
        self.augment = augment
        self.to_tensor = ToTensor()

    def trans(self, img_20x, img_5x, label):
        if random.random() > 0.5:
            img_20x = transforms.functional.hflip(img_20x)
            img_5x = transforms.functional.hflip(img_5x)
            label = torch.flip(label, dims=[2])
        if random.random() > 0.5:
            img_20x = transforms.functional.vflip(img_20x)
            img_5x = transforms.functional.vflip(img_5x)
            label = torch.flip(label, dims=[1])
        return img_20x, img_5x, label

    def __len__(self):
        return len(self.img_20x_paths)

    def __getitem__(self, idx):
        img_20x = self.to_tensor(Image.open(self.img_20x_paths[idx]).convert('RGB'))
        img_5x = self.to_tensor(Image.open(self.img_5x_paths[idx]).convert('RGB'))
        label = torch.from_numpy(np.load(self.label_paths[idx])).float()

        if self.augment:
            img_20x, img_5x, label = self.trans(img_20x, img_5x, label)

        return img_20x, img_5x, label

# ===== 데이터 수집 =====
img_20x_list = sorted(glob(f'{data_dir}/image/**/*.tiff', recursive=True))
img_5x_list = [p.replace('/image/', '/image_5x/') for p in img_20x_list]
label_list = [p.replace('/image/', '/label/').replace('.tiff', '.npy') for p in img_20x_list]

# 모든 파일 존재 확인
valid = [(a, b, c) for a, b, c in zip(img_20x_list, img_5x_list, label_list)
         if os.path.exists(b) and os.path.exists(c)]
img_20x_list = [v[0] for v in valid]
img_5x_list = [v[1] for v in valid]
label_list = [v[2] for v in valid]

print(f'Total valid patches: {len(img_20x_list)}')

# ===== Train/Val 분할 (인덱스 기반) =====
indices = list(range(len(img_20x_list)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_20x = [img_20x_list[i] for i in train_idx]
train_5x = [img_5x_list[i] for i in train_idx]
train_labels = [label_list[i] for i in train_idx]

val_20x = [img_20x_list[i] for i in val_idx]
val_5x = [img_5x_list[i] for i in val_idx]
val_labels = [label_list[i] for i in val_idx]

print(f'Train: {len(train_20x)}, Val: {len(val_20x)}')

# ===== DataLoader =====
train_dataset = MultiScaleDataset(train_20x, train_5x, train_labels, augment=True)
val_dataset = MultiScaleDataset(val_20x, val_5x, val_labels, augment=False)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=4, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True, num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_dataloader)}, Val batches: {len(val_dataloader)}')

In [ ]:
# ===== Multi-Scale 모델 정의 =====
# 20x encoder: 세포 형태 등 fine-grained 특징 추출
# 5x encoder: 조직 구조 등 contextual 특징 추출
# 두 encoder의 feature를 stage별로 합산 → 단일 decoder로 복원

class MultiScaleGeneExpressionModel(nn.Module):
    def __init__(self, encoder_name='tu-convnext_tiny', num_genes=15):
        super().__init__()
        self.encoder_20x = smp.encoders.get_encoder(
            encoder_name, in_channels=3, weights='imagenet'
        )
        self.encoder_5x = smp.encoders.get_encoder(
            encoder_name, in_channels=3, weights='imagenet'
        )

        encoder_channels = self.encoder_20x.out_channels

        self.decoder = smp.decoders.unet.decoder.UnetDecoder(
            encoder_channels=encoder_channels,
            decoder_channels=(256, 128, 64, 32, 16),
            n_blocks=5,
        )

        self.seg_head = smp.base.SegmentationHead(
            in_channels=16, out_channels=num_genes, activation=None
        )

    def forward(self, x_20x, x_5x):
        feat_20x = self.encoder_20x(x_20x)
        feat_5x = self.encoder_5x(x_5x)

        fused = [f20 + f5 for f20, f5 in zip(feat_20x, feat_5x)]

        out = self.decoder(fused)
        out = self.seg_head(out)
        out = F.interpolate(out, size=(heatmap_size, heatmap_size),
                            mode='bilinear', align_corners=False)
        out = torch.sigmoid(out)
        return out


# ===== Loss 정의: Huber + Pearson Correlation Loss =====
class HuberPearsonLoss(nn.Module):
    def __init__(self, huber_weight=1.0, pcc_weight=1.0, huber_delta=1.0):
        super().__init__()
        self.huber_weight = huber_weight
        self.pcc_weight = pcc_weight
        self.huber = nn.HuberLoss(delta=huber_delta)

    def pearson_loss(self, pred, target):
        B, C, H, W = pred.shape
        pred = pred.view(B, C, -1)
        target = target.view(B, C, -1)

        pred_c = pred - pred.mean(dim=2, keepdim=True)
        target_c = target - target.mean(dim=2, keepdim=True)

        cov = (pred_c * target_c).sum(dim=2)
        pred_std = pred_c.pow(2).sum(dim=2).sqrt()
        target_std = target_c.pow(2).sum(dim=2).sqrt()

        pcc = cov / (pred_std * target_std + 1e-8)
        return 1 - pcc.mean()

    def forward(self, pred, target):
        loss_huber = self.huber(pred, target)
        loss_pcc = self.pearson_loss(pred, target)
        return self.huber_weight * loss_huber + self.pcc_weight * loss_pcc


model = MultiScaleGeneExpressionModel(
    encoder_name='tu-convnext_tiny', num_genes=num_genes
).to(device)

criterion = HuberPearsonLoss(huber_weight=1.0, pcc_weight=1.0)
optimizer = optim.Adam(model.parameters(), lr=lr)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

In [ ]:
# ===== Pearson Correlation Coefficient =====
def pearson_corr(pred, target):
    B, C, H, W = pred.shape
    pred = pred.view(B, C, -1)
    target = target.view(B, C, -1)

    pred_mean = pred.mean(dim=2, keepdim=True)
    target_mean = target.mean(dim=2, keepdim=True)

    pred_c = pred - pred_mean
    target_c = target - target_mean

    cov = (pred_c * target_c).sum(dim=2)
    pred_std = pred_c.pow(2).sum(dim=2).sqrt()
    target_std = target_c.pow(2).sum(dim=2).sqrt()

    pcc = cov / (pred_std * target_std + 1e-8)
    return pcc.mean(dim=0)  # (15,)

# ===== 학습 루프 =====
train_loss_list = []
val_loss_list = []
train_pcc_list = []
val_pcc_list = []
MIN_loss = float('inf')

for epoch in range(num_epochs):
    # ===== Train =====
    model.train()
    train_bar = tqdm(train_dataloader, desc=f'Train {epoch+1}/{num_epochs}')
    running_loss = 0.0
    running_pcc = torch.zeros(num_genes)
    count = 0

    for x_20x, x_5x, y in train_bar:
        x_20x = x_20x.to(device).float()
        x_5x = x_5x.to(device).float()
        y = y.to(device).float()
        count += 1

        optimizer.zero_grad()
        pred = model(x_20x, x_5x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        with torch.no_grad():
            running_pcc += pearson_corr(pred, y).cpu()

        avg_pcc = running_pcc.mean().item() / count
        train_bar.set_postfix(loss=f'{running_loss/count:.4f}', pcc=f'{avg_pcc:.4f}')

    train_loss_list.append(running_loss / count)
    train_pcc_list.append((running_pcc / count).mean().item())

    # ===== Validation =====
    model.eval()
    val_bar = tqdm(val_dataloader, desc=f'Val   {epoch+1}/{num_epochs}')
    val_loss = 0.0
    val_pcc = torch.zeros(num_genes)
    count = 0

    with torch.no_grad():
        for x_20x, x_5x, y in val_bar:
            x_20x = x_20x.to(device).float()
            x_5x = x_5x.to(device).float()
            y = y.to(device).float()
            count += 1

            pred = model(x_20x, x_5x)
            loss = criterion(pred, y)

            val_loss += loss.item()
            val_pcc += pearson_corr(pred, y).cpu()

            avg_pcc = val_pcc.mean().item() / count
            val_bar.set_postfix(loss=f'{val_loss/count:.4f}', pcc=f'{avg_pcc:.4f}')

    val_loss_list.append(val_loss / count)
    val_pcc_list.append((val_pcc / count).mean().item())

    # ===== Best Model 저장 =====
    if MIN_loss > val_loss / count:
        torch.save(model.state_dict(), f'{model_path}train_multiscale_best.pt')
        MIN_loss = val_loss / count
        print(f'  -> Best model saved (val_loss: {MIN_loss:.6f})')

    # ===== 주기적 시각화 =====
    if epoch % 50 == 5:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        axes[0].set_title('Loss')
        axes[0].plot(train_loss_list, label='train')
        axes[0].plot(val_loss_list, label='val')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('MSE Loss')
        axes[0].legend()

        axes[1].set_title('Pearson Correlation')
        axes[1].plot(train_pcc_list, label='train')
        axes[1].plot(val_pcc_list, label='val')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('PCC')
        axes[1].set_ylim([0, 1])
        axes[1].legend()

        plt.tight_layout()
        plt.show()

# ===== 최종 그래프 =====
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].set_title('Loss')
axes[0].plot(train_loss_list, label='train')
axes[0].plot(val_loss_list, label='val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

axes[1].set_title('Pearson Correlation')
axes[1].plot(train_pcc_list, label='train')
axes[1].plot(val_pcc_list, label='val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('PCC')
axes[1].set_ylim([0, 1])
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nBest val loss: {MIN_loss:.6f}')
print(f'batch_size={batch_size}, img_size={img_size}, lr={lr}')

In [ ]:
# ===== 검증 시각화 =====
model.load_state_dict(torch.load(f'{model_path}train_multiscale_best.pt', map_location=device))
model.eval()

val_iter = iter(val_dataloader)
x_20x, x_5x, y_test = next(val_iter)
x_20x = x_20x.to(device).float()
x_5x = x_5x.to(device).float()

with torch.no_grad():
    pred = model(x_20x, x_5x)

x20_np = x_20x.cpu().numpy()
x5_np = x_5x.cpu().numpy()
y_np = y_test.cpu().numpy()
pred_np = pred.cpu().numpy()

# 샘플 2개, gene 5개 시각화
show_genes = [0, 4, 8, 11, 14]  # EPCAM, COL1A1, CD3D, MS4A1, PECAM1
n_samples = min(2, x20_np.shape[0])

for sample_idx in range(n_samples):
    fig, axes = plt.subplots(3, len(show_genes) + 1, figsize=(3 * (len(show_genes) + 1), 9))

    # Row 0: 20x + 5x 입력
    axes[0, 0].imshow(np.transpose(x20_np[sample_idx], (1, 2, 0)))
    axes[0, 0].set_title('20x Input')
    axes[0, 0].axis('off')

    # Row 1: GT
    axes[1, 0].imshow(np.transpose(x5_np[sample_idx], (1, 2, 0)))
    axes[1, 0].set_title('5x Context')
    axes[1, 0].axis('off')

    # Row 2: Pred
    axes[2, 0].imshow(np.transpose(x20_np[sample_idx], (1, 2, 0)))
    axes[2, 0].set_title('20x Input')
    axes[2, 0].axis('off')

    for i, gi in enumerate(show_genes):
        axes[0, i + 1].imshow(y_np[sample_idx, gi], cmap='hot', vmin=0, vmax=1)
        axes[0, i + 1].set_title(f'GT: {marker_genes[gi]}')
        axes[0, i + 1].axis('off')

        axes[1, i + 1].imshow(pred_np[sample_idx, gi], cmap='hot', vmin=0, vmax=1)
        axes[1, i + 1].set_title(f'Pred: {marker_genes[gi]}')
        axes[1, i + 1].axis('off')

        # Difference map
        diff = np.abs(y_np[sample_idx, gi] - pred_np[sample_idx, gi])
        axes[2, i + 1].imshow(diff, cmap='coolwarm', vmin=0, vmax=0.5)
        axes[2, i + 1].set_title(f'Diff: {marker_genes[gi]}')
        axes[2, i + 1].axis('off')

    fig.suptitle(f'Sample {sample_idx + 1}: GT (top) / Pred (mid) / |Diff| (bot)', fontsize=14)
    plt.tight_layout()
    plt.show()

# ===== Gene별 PCC/MAE 결과 =====
print('\n===== Gene-wise Evaluation on Validation Set =====')
all_pcc = torch.zeros(num_genes)
all_mae = torch.zeros(num_genes)
count = 0

with torch.no_grad():
    for x_20x, x_5x, y in tqdm(val_dataloader, desc='Evaluating'):
        x_20x = x_20x.to(device).float()
        x_5x = x_5x.to(device).float()
        y = y.to(device).float()
        pred = model(x_20x, x_5x)
        count += 1

        all_pcc += pearson_corr(pred, y).cpu()
        all_mae += (pred - y).abs().mean(dim=(0, 2, 3)).cpu()

all_pcc /= count
all_mae /= count

print(f'{"Gene":>10s} | {"PCC":>8s} | {"MAE":>8s}')
print('-' * 32)
for i, gene in enumerate(marker_genes):
    print(f'{gene:>10s} | {all_pcc[i].item():8.4f} | {all_mae[i].item():8.4f}')
print('-' * 32)
print(f'{"Mean":>10s} | {all_pcc.mean().item():8.4f} | {all_mae.mean().item():8.4f}')